***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [6. 成像中的去卷积](6_0_introduction.ipynb)
    * 上一节： [6.1 天空模型](6_1_sky_models.ipynb)
    * 下一节： [6.3 CLEAN 的实现方式](6_3_clean_flavours.ipynb)

***

导入标准模块:

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['figure.max_open_warning'] = 0

NOTEBOOK_DIR = Path('6_Deconvolution') if Path('6_Deconvolution').exists() else Path('.')
NOTEBOOK_DIR = NOTEBOOK_DIR.resolve()
STYLE_PATH = NOTEBOOK_DIR.parent / 'style' / 'course.css'
TOGGLE_PATH = NOTEBOOK_DIR.parent / 'style' / 'code_toggle.html'

try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None

if STYLE_PATH.exists():
    try:
        HTML(STYLE_PATH.read_text(encoding='utf-8'))
    except Exception:
        pass

本节以下示例全部使用 notebook 内生成的合成观测，因此不再依赖 `astropy`、`aplpy` 或外部 FITS 文件。这样做的目的，是把 `CLEAN` 的核心逻辑拆开来讲：我们已知真实天空、已知脏波束、也知道加性噪声的大小，于是就能清楚地看到算法为什么有效，以及它在什么条件下会开始失效。

In [ ]:
if TOGGLE_PATH.exists():
    try:
        HTML(TOGGLE_PATH.read_text(encoding='utf-8'))
    except Exception:
        pass

***

## 6.2 点源的迭代去卷积（CLEAN）<a id='deconv:sec:clean'></a>

如果我们接受“天空可以由一组离散点源近似表示”这一先验，那么去卷积问题就不必再理解成一次性地“求一个逆算子”，而可以改写成一个逐步构造模型的过程。对射电干涉测量而言，这正是 `CLEAN` 的思想起点。

在脏图语言中，我们希望处理的是

$$
I^{D}(l,m) = \mathrm{PSF}(l,m) \ast I_{\mathrm{true}}(l,m) + N(l,m),
$$

其中 $I^{D}$ 是脏图，$\mathrm{PSF}$ 是阵列的脏波束，$I_{\mathrm{true}}$ 是真实天空，$N$ 是噪声与各种未建模误差。困难在于：$\mathrm{PSF}$ 往往带有强旁瓣，而观测噪声又会在逆运算中被显著放大。因此，`CLEAN` 采取的不是“直接反卷积”，而是“每次只解释残图里最显著的一小部分结构”。

从现代优化语言看，`CLEAN` 很像一种稀疏贪婪逼近：字典元素是“位于不同位置的 PSF 副本”，而我们要做的，是用尽可能少的分量去解释脏图中的主要信号。

In [ ]:
def generalGauss2d(x0, y0, sigmax, sigmay, amp=1.0, theta=0.0):
    rtheta = np.deg2rad(theta)
    a = (np.cos(rtheta) ** 2.0) / (2.0 * sigmax ** 2.0) + (np.sin(rtheta) ** 2.0) / (2.0 * sigmay ** 2.0)
    b = -(np.sin(2.0 * rtheta)) / (4.0 * sigmax ** 2.0) + (np.sin(2.0 * rtheta)) / (4.0 * sigmay ** 2.0)
    c = (np.sin(rtheta) ** 2.0) / (2.0 * sigmax ** 2.0) + (np.cos(rtheta) ** 2.0) / (2.0 * sigmay ** 2.0)
    return lambda x, y: amp * np.exp(-1.0 * (a * (x - x0) ** 2.0 - 2.0 * b * (x - x0) * (y - y0) + c * (y - y0) ** 2.0))



def centered_gaussian(n, fwhm_x, fwhm_y=None, theta=0.0, amp=1.0):
    if fwhm_y is None:
        fwhm_y = fwhm_x
    sigma_x = fwhm_x / (2.0 * np.sqrt(2.0 * np.log(2.0)))
    sigma_y = fwhm_y / (2.0 * np.sqrt(2.0 * np.log(2.0)))
    yy, xx = np.mgrid[0:n, 0:n].astype(float)
    arr = generalGauss2d((n - 1) / 2.0, (n - 1) / 2.0, sigma_x, sigma_y, amp=amp, theta=theta)(xx, yy)
    return arr / arr.max()



def fft_convolve_same(image, kernel):
    return np.real(
        np.fft.fftshift(
            np.fft.ifft2(
                np.fft.fft2(np.fft.ifftshift(image)) * np.fft.fft2(np.fft.ifftshift(kernel))
            )
        )
    )



def make_uv_dirty_beam(n, n_pairs=220, radius_frac=0.33, seed=7):
    rng = np.random.default_rng(seed)
    center = n // 2
    radius = radius_frac * n
    angles = rng.uniform(0.0, 2.0 * np.pi, n_pairs)
    radii = radius * np.sqrt(rng.uniform(0.03, 1.0, n_pairs))
    u = np.rint(radii * np.cos(angles)).astype(int)
    v = np.rint(radii * np.sin(angles)).astype(int)
    keep = (np.abs(u) < n // 2 - 2) & (np.abs(v) < n // 2 - 2) & ((u != 0) | (v != 0))
    half = np.column_stack([u[keep], v[keep]])
    coords = np.vstack([half, -half, np.array([[0, 0]])])

    sampling = np.zeros((n, n), dtype=float)
    sampling[coords[:, 1] + center, coords[:, 0] + center] = 1.0

    psf = np.real(np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(sampling))))
    psf /= psf.max()
    return sampling, psf



def make_point_sky(n, catalog):
    sky = np.zeros((n, n), dtype=float)
    for flux, x, y in catalog:
        sky[int(y), int(x)] += flux
    return sky



def estimate_clean_beam(psf, radius=6):
    n = psf.shape[0]
    cy = cx = n // 2
    yy, xx = np.mgrid[0:n, 0:n].astype(float)
    rr2 = (xx - cx) ** 2 + (yy - cy) ** 2
    weights = np.clip(psf, 0.0, None) * (rr2 <= radius ** 2)
    weights_sum = weights.sum()

    if weights_sum <= 0:
        return centered_gaussian(n, 3.0), 3.0, 3.0, 0.0

    x_mean = (weights * xx).sum() / weights_sum
    y_mean = (weights * yy).sum() / weights_sum
    dx = xx - x_mean
    dy = yy - y_mean
    cov_xx = (weights * dx * dx).sum() / weights_sum
    cov_yy = (weights * dy * dy).sum() / weights_sum
    cov_xy = (weights * dx * dy).sum() / weights_sum
    cov = np.array([[cov_xx, cov_xy], [cov_xy, cov_yy]])
    eigvals, eigvecs = np.linalg.eigh(cov)
    eigvals = np.clip(eigvals, 1e-6, None)

    major_sigma = np.sqrt(eigvals[1])
    minor_sigma = np.sqrt(eigvals[0])
    theta = np.rad2deg(np.arctan2(eigvecs[1, 1], eigvecs[0, 1]))
    fwhm_x = 2.0 * np.sqrt(2.0 * np.log(2.0)) * major_sigma
    fwhm_y = 2.0 * np.sqrt(2.0 * np.log(2.0)) * minor_sigma
    clean_beam = centered_gaussian(n, fwhm_x, fwhm_y, theta=theta)
    return clean_beam, fwhm_x, fwhm_y, theta



def hogbom_clean(dirty, psf, gain=0.1, niter=2000, threshold=0.0, clean_beam=None, capture_iters=(0,)):
    capture_iters = set(capture_iters)
    residual = dirty.copy()
    model = np.zeros_like(dirty)
    center = np.array(psf.shape) // 2
    history = []
    components = []
    states = {}

    def capture(iteration):
        model_image = fft_convolve_same(model, clean_beam) if clean_beam is not None else model.copy()
        restored = model_image + residual
        states[iteration] = {
            'model': model.copy(),
            'residual': residual.copy(),
            'model_image': model_image.copy(),
            'restored': restored.copy(),
        }

    capture(0)
    history.append((0, np.max(np.abs(residual)), np.sqrt(np.mean(residual ** 2)), model.sum()))
    last_iter = 0

    for iteration in range(1, niter + 1):
        iy, ix = np.unravel_index(np.argmax(np.abs(residual)), residual.shape)
        peak = residual[iy, ix]
        if np.abs(peak) <= threshold:
            break

        amp = gain * peak
        components.append((iteration, ix, iy, peak, amp))
        model[iy, ix] += amp
        shifted_psf = np.roll(np.roll(psf, iy - center[0], axis=0), ix - center[1], axis=1)
        residual = residual - amp * shifted_psf
        last_iter = iteration
        history.append((iteration, np.max(np.abs(residual)), np.sqrt(np.mean(residual ** 2)), model.sum()))

        if iteration in capture_iters:
            capture(iteration)

    if last_iter not in states:
        capture(last_iter)

    model_image = fft_convolve_same(model, clean_beam) if clean_beam is not None else model.copy()
    restored = model_image + residual

    return {
        'model': model,
        'residual': residual,
        'model_image': model_image,
        'restored': restored,
        'history': np.array(history, dtype=float),
        'components': components,
        'states': states,
        'iterations': last_iter,
    }

### 6.2.1 采样、PSF 与脏图<a id='deconv:sec:sampling'></a>

In [ ]:
n = 128
trueCatalog = [
    (1.00, 26, 30),
    (0.72, 37, 96),
    (0.55, 64, 24),
    (0.48, 83, 74),
    (0.32, 104, 46),
    (0.26, 91, 103),
]

trueSky = make_point_sky(n, trueCatalog)
samplingFunc, dirtyBeam = make_uv_dirty_beam(n=n, n_pairs=220, radius_frac=0.33, seed=7)
cleanBeam, cleanBeamFwhmX, cleanBeamFwhmY, cleanBeamTheta = estimate_clean_beam(dirtyBeam, radius=7)

noiseSigma = 0.018
rng = np.random.default_rng(22)
dirtyImage = fft_convolve_same(trueSky, dirtyBeam) + rng.normal(scale=noiseSigma, size=(n, n))

print(f'真实源总流量                           = {trueSky.sum():.3f}')
print(f'脏图峰值                               = {dirtyImage.max():.3f}')
print(f'加入的像素噪声 sigma                   = {noiseSigma:.3f}')
print(f'估计得到的 clean beam FWHM             = {cleanBeamFwhmX:.2f} x {cleanBeamFwhmY:.2f} pixel')
print(f'估计得到的 clean beam 位置角            = {cleanBeamTheta:.1f} deg')

fig, axes = plt.subplots(2, 2, figsize=(13.5, 11))

im0 = axes[0, 0].imshow(trueSky, origin='lower', cmap='viridis', vmin=0.0, vmax=1.05)
for flux, x, y in trueCatalog:
    axes[0, 0].plot(x, y, marker='x', color='white', markersize=7, mew=1.3)
axes[0, 0].set_title('真实点源天空模型')
axes[0, 0].set_xlabel('Pixel')
axes[0, 0].set_ylabel('Pixel')
fig.colorbar(im0, ax=axes[0, 0], shrink=0.8)

im1 = axes[0, 1].imshow(samplingFunc, origin='lower', cmap='gray_r')
axes[0, 1].set_title('uv 采样函数')
axes[0, 1].set_xlabel('u pixel')
axes[0, 1].set_ylabel('v pixel')
fig.colorbar(im1, ax=axes[0, 1], shrink=0.8)

im2 = axes[1, 0].imshow(dirtyBeam, origin='lower', cmap='coolwarm', vmin=-0.25, vmax=1.0)
axes[1, 0].set_title('对应的脏波束（PSF）')
axes[1, 0].set_xlabel('Pixel')
axes[1, 0].set_ylabel('Pixel')
fig.colorbar(im2, ax=axes[1, 0], shrink=0.8)

im3 = axes[1, 1].imshow(dirtyImage, origin='lower', cmap='viridis', vmin=-0.12, vmax=1.08)
axes[1, 1].set_title('卷积并加噪后的脏图')
axes[1, 1].set_xlabel('Pixel')
axes[1, 1].set_ylabel('Pixel')
fig.colorbar(im3, ax=axes[1, 1], shrink=0.8)

plt.tight_layout()

*图：左上是真实点源天空；右上是离散的 `uv` 采样函数；左下是由该采样函数决定的脏波束；右下是脏图。这个例子已经把 `CLEAN` 的工作对象摆清楚了：我们看到的不是“真实天空本身”，而是“真实天空与复杂 PSF 卷积后的结果，再叠加噪声”。*

In [ ]:
psfTransfer = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(dirtyBeam)))
smallModeFraction = np.mean(np.abs(psfTransfer) < 0.03 * np.max(np.abs(psfTransfer)))
regularization = 0.002 * np.max(np.abs(psfTransfer))

inverseFilterImage = np.real(
    np.fft.fftshift(
        np.fft.ifft2(
            np.fft.fft2(np.fft.ifftshift(dirtyImage)) / (psfTransfer + regularization)
        )
    )
)

print(f'|PSF~| 低于最大值 3% 的傅里叶模式比例      = {smallModeFraction:.3f}')
print(f'脏图 RMS                                 = {np.sqrt(np.mean(dirtyImage ** 2)):.4f}')
print(f'逆滤波图像 RMS                           = {np.sqrt(np.mean(inverseFilterImage ** 2)):.4f}')
print(f'逆滤波图像的最大绝对像素值               = {np.max(np.abs(inverseFilterImage)):.4f}')

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))

im0 = axes[0].imshow(
    np.log10(np.abs(psfTransfer) / np.max(np.abs(psfTransfer)) + 1e-6),
    origin='lower',
    cmap='magma',
    vmin=-6,
    vmax=0,
)
axes[0].set_title(r'$\log_{10}|\widetilde{\mathrm{PSF}}|$')
axes[0].set_xlabel('u pixel')
axes[0].set_ylabel('v pixel')
fig.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = axes[1].imshow(inverseFilterImage, origin='lower', cmap='coolwarm', vmin=-1.0, vmax=1.0)
axes[1].set_title('直接逆滤波的结果')
axes[1].set_xlabel('Pixel')
axes[1].set_ylabel('Pixel')
fig.colorbar(im1, ax=axes[1], shrink=0.8)

im2 = axes[2].imshow(trueSky, origin='lower', cmap='viridis', vmin=0.0, vmax=1.05)
axes[2].set_title('真实天空（供对照）')
axes[2].set_xlabel('Pixel')
axes[2].set_ylabel('Pixel')
fig.colorbar(im2, ax=axes[2], shrink=0.8)

plt.tight_layout()

这个例子说明，去卷积绝不是“把脏图除以 PSF”这么简单。傅里叶域里有大量模式的传递函数幅度非常小，一旦直接相除，哪怕只是图像中的微小噪声，也会被这些近零模式剧烈放大。结果就是：你得到的不是干净的天空模型，而是一个充满振铃和噪声放大的不稳定解。

因此，`CLEAN` 的关键不是发明某个神奇的逆核，而是在一个合理的天空先验下，逐步解释脏图里最显著的结构。

### 6.2.2 点源先验与 Högbom CLEAN<a id='deconv:sec:hogbom'></a>

在最经典的 Högbom `CLEAN` 中，我们假设天空可以由一组点源组成。这样一来，每一次迭代就可以非常明确地写成：

1. 在当前残图 $R_n$ 中找到绝对值最大的像素位置 $(l_p, m_p)$；
2. 把该像素的一部分流量加到 `CLEAN` 组分模型里；
3. 从残图中减去一个平移到该位置的 PSF 副本；
4. 重复上述过程，直到残图峰值低于阈值，或者达到最大迭代次数。

公式上，若循环增益记为 $\gamma$，则

$$
M_{n+1}(l,m) = M_n(l,m) + \gamma \, R_n(l_p,m_p) \, \delta(l-l_p, m-m_p),
$$

$$
R_{n+1}(l,m) = R_n(l,m) - \gamma \, R_n(l_p,m_p) \, \mathrm{PSF}(l-l_p, m-m_p).
$$

这里的 $\gamma$ 通常取一个小于 1 的数。它的作用并不是“让算法更慢”这么简单，而是在脏波束旁瓣存在时，避免一次性减得过头。等到迭代结束后，再把点源组分模型与一个理想化的 `clean beam` 卷积，并加回最终残图，得到

$$
I_{\mathrm{restored}} = B_{\mathrm{clean}} \ast M + R_{\mathrm{final}}.
$$

下面先看前几个 `CLEAN` 组分是怎样一步步被选出来的。

In [ ]:
referenceGain = 0.18
referenceThreshold = 3.0 * noiseSigma
referenceRun = hogbom_clean(
    dirtyImage,
    dirtyBeam,
    gain=referenceGain,
    niter=2000,
    threshold=referenceThreshold,
    clean_beam=cleanBeam,
    capture_iters=(10, 40, 120, 200),
)

print(f'参考运行的循环增益                    = {referenceGain:.2f}')
print(f'参考运行的停止阈值                    = {referenceThreshold / noiseSigma:.1f} sigma')
print(f'达到停止条件时的迭代次数              = {referenceRun["iterations"]}')
print(f'最终残图峰值                           = {np.max(np.abs(referenceRun["residual"])):.4f}')
print(f'最终残图 RMS                            = {np.sqrt(np.mean(referenceRun["residual"] ** 2)):.4f}')

print()
print('前 12 个 CLEAN 组分：')
print(' iter   xpix   ypix   residual peak   added component')
for iteration, xpix, ypix, peak, amp in referenceRun['components'][:12]:
    print(f' {iteration:4d}   {xpix:4d}   {ypix:4d}      {peak:8.4f}         {amp:8.4f}')

可以看到，前几个组分并不是在所有真实源之间“平均分配”的。算法会先反复挑选最亮源附近的峰值，再逐渐转向次亮源。这正是循环增益小于 1 时的自然行为：每次只减去峰值的一部分，因此同一个亮源往往会在多次迭代中不断累加组分。这个机制让 `CLEAN` 能够在强旁瓣存在时保持相对稳定。

In [ ]:
finalIter = referenceRun['iterations']

fig, axes = plt.subplots(2, 3, figsize=(17, 11))

im0 = axes[0, 0].imshow(dirtyImage, origin='lower', cmap='viridis', vmin=-0.12, vmax=1.08)
axes[0, 0].set_title('初始脏图')
axes[0, 0].set_xlabel('Pixel')
axes[0, 0].set_ylabel('Pixel')
fig.colorbar(im0, ax=axes[0, 0], shrink=0.8)

im1 = axes[0, 1].imshow(referenceRun['states'][10]['residual'], origin='lower', cmap='coolwarm', vmin=-0.18, vmax=0.18)
axes[0, 1].set_title('第 10 次迭代后的残图')
axes[0, 1].set_xlabel('Pixel')
axes[0, 1].set_ylabel('Pixel')
fig.colorbar(im1, ax=axes[0, 1], shrink=0.8)

im2 = axes[0, 2].imshow(referenceRun['states'][40]['residual'], origin='lower', cmap='coolwarm', vmin=-0.18, vmax=0.18)
axes[0, 2].set_title('第 40 次迭代后的残图')
axes[0, 2].set_xlabel('Pixel')
axes[0, 2].set_ylabel('Pixel')
fig.colorbar(im2, ax=axes[0, 2], shrink=0.8)

im3 = axes[1, 0].imshow(referenceRun['model'], origin='lower', cmap='inferno', vmin=0.0, vmax=0.25)
axes[1, 0].set_title('最终 CLEAN 组分模型')
axes[1, 0].set_xlabel('Pixel')
axes[1, 0].set_ylabel('Pixel')
fig.colorbar(im3, ax=axes[1, 0], shrink=0.8)

im4 = axes[1, 1].imshow(referenceRun['restored'], origin='lower', cmap='viridis', vmin=-0.08, vmax=0.95)
axes[1, 1].set_title('复原图像')
axes[1, 1].set_xlabel('Pixel')
axes[1, 1].set_ylabel('Pixel')
fig.colorbar(im4, ax=axes[1, 1], shrink=0.8)

im5 = axes[1, 2].imshow(referenceRun['residual'], origin='lower', cmap='coolwarm', vmin=-0.12, vmax=0.12)
axes[1, 2].set_title(f'最终残图（{finalIter} 次迭代）')
axes[1, 2].set_xlabel('Pixel')
axes[1, 2].set_ylabel('Pixel')
fig.colorbar(im5, ax=axes[1, 2], shrink=0.8)

plt.tight_layout()

print(f'最终 CLEAN 分量总流量                   = {referenceRun["model"].sum():.4f}')
print(f'真实总流量                               = {trueSky.sum():.4f}')
print(f'非零 CLEAN 分量个数                      = {np.count_nonzero(np.abs(referenceRun["model"]) > 1e-12)}')
print()
print('按每个真实源附近 2 像素半径汇总的 CLEAN 流量：')
print('  xpix   ypix   true flux   recovered flux')
yy, xx = np.mgrid[0:n, 0:n]
for flux, xpix, ypix in trueCatalog:
    localMask = (xx - xpix) ** 2 + (yy - ypix) ** 2 <= 2 ** 2
    recoveredFlux = referenceRun['model'][localMask].sum()
    print(f'  {xpix:4d}   {ypix:4d}     {flux:7.3f}        {recoveredFlux:7.3f}')

*图：上排显示残图如何在迭代中逐步“削去”主旁瓣结构；下排依次是最终点源组分模型、复原图像与最终残图。对于这类以离散点源为主、且源间距较大的天空，Högbom `CLEAN` 已经能把大部分显著结构从脏图里剥离出来，并给出相当合理的恢复结果。*

### 6.2.3 循环增益与停止阈值<a id='deconv:sec:clean_params'></a>

In [ ]:
gainResults = {}
for gain in [0.05, 0.18, 0.45, 0.80]:
    gainResults[gain] = hogbom_clean(
        dirtyImage,
        dirtyBeam,
        gain=gain,
        niter=1500,
        threshold=referenceThreshold,
        clean_beam=cleanBeam,
    )

fig, ax = plt.subplots(figsize=(8.5, 5.5))
for gain, run in gainResults.items():
    history = run['history']
    ax.plot(history[:, 0], history[:, 1] / noiseSigma, linewidth=2, label=f'gain = {gain:.2f}')
ax.axhline(referenceThreshold / noiseSigma, color='k', linestyle='--', linewidth=1.2, label='3 sigma threshold')
ax.set_yscale('log')
ax.set_xlabel('Iteration')
ax.set_ylabel(r'Max residual / noise sigma')
ax.set_title('不同循环增益下的收敛轨迹')
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()

print('不同循环增益（停止阈值固定为 3 sigma）：')
print(' gain   iter   final peak/sigma   residual RMS   ncomp   model flux')
for gain, run in gainResults.items():
    history = run['history']
    finalPeakSigma = history[-1, 1] / noiseSigma
    residualRms = np.sqrt(np.mean(run['residual'] ** 2))
    ncomp = np.count_nonzero(np.abs(run['model']) > 1e-12)
    print(f' {gain:4.2f}   {run["iterations"]:4d}        {finalPeakSigma:7.3f}        {residualRms:8.4f}    {ncomp:5d}    {run["model"].sum():8.4f}')

print()
print('不同停止阈值（循环增益固定为 0.18）：')
print(' threshold   iter   final peak/sigma   residual RMS   ncomp   model flux')
for thresholdSigma in [5.0, 3.0, 2.0]:
    run = hogbom_clean(
        dirtyImage,
        dirtyBeam,
        gain=referenceGain,
        niter=1500,
        threshold=thresholdSigma * noiseSigma,
        clean_beam=cleanBeam,
    )
    history = run['history']
    finalPeakSigma = history[-1, 1] / noiseSigma
    residualRms = np.sqrt(np.mean(run['residual'] ** 2))
    ncomp = np.count_nonzero(np.abs(run['model']) > 1e-12)
    print(f'   {thresholdSigma:3.0f} sigma   {run["iterations"]:4d}        {finalPeakSigma:7.3f}        {residualRms:8.4f}    {ncomp:5d}    {run["model"].sum():8.4f}')

这里的结果非常典型：

* 循环增益太小，例如 `0.05`，算法会比较稳，但收敛明显变慢；
* 循环增益取中等值，例如 `0.1` 到 `0.2`，通常是比较平衡的工作区间；
* 循环增益太大时，残图峰值不一定单调稳定下降，还会引入更多分量，把旁瓣和噪声一并“吃进去”；
* 停止阈值太高会导致欠清理，亮源周围仍残留明显旁瓣；停止阈值太低又会把算法拖进噪声主导区，分量数暴涨，模型总流量也开始偏大。

因此，在真实数据处理中，`CLEAN` 的关键从来不是“多跑一些迭代就更好”，而是找到一个与噪声水平、波束结构和科学目标相匹配的停止点。

这一节的目标，是先把点源 `CLEAN` 的核心思路建立起来：它为什么不做直接逆滤波、为什么要引入循环增益、为什么复原图像要把 `clean beam` 和最终残图组合起来。下一节会继续比较 `CLEAN` 的几种实现方式，包括 Högbom、Clark 与 Cotton-Schwab 的关系；再下一节则专门讨论残图诊断、动态范围与图像质量问题。

***

* 下一节： [6.3 CLEAN 的实现方式](6_3_clean_flavours.ipynb)